# 🚀 **BERT Sentiment Lab - GPU Tesla T4 Backend Server (Google Colab)**

Notebook ini digunakan untuk mengaktifkan **Backend FastAPI** menggunakan akselerasi **NVIDIA Tesla T4 GPU** di Google Colab.

### 📌 **Langkah Persiapan:**
1. Pastikan Runtime Colab menggunakan GPU T4: **Menu Runtime -> Change runtime type -> T4 GPU**.
2. Jalankan cell per sel berturut-turut di bawah ini.
3. Sediakan **1 Free Ngrok Static Domain** dari dashboard.ngrok.com agar URL **TIDAK PERNAH BERUBAH** setiap kali Colab di-restart!

In [ ]:
# ==========================================
# 1. Verifikasi Keberadaan GPU (Tesla T4)
# ==========================================
import torch

if not torch.cuda.is_available():
    raise SystemError("❌ GPU (CUDA) tidak terdeteksi! Harap ubah runtime Google Colab Anda ke GPU Tesla T4 (Menu: Runtime > Change runtime type > T4 GPU).")

print(f"✅ GPU AKTIF: {torch.cuda.get_device_name(0)}")
print(f"   VRAM Tersedia: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# ==========================================
# 2. Clone Repositori & Install Dependensi
# ==========================================
import os

REPO_URL = "https://github.com/syafiqhsn/BERTsentiment.lab.git"
REPO_DIR = "BERTsentiment.lab"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL}

%cd {REPO_DIR}

print("Installing dependencies...")
!pip install -q -r requirements.txt pyngrok gdown

## 🧪 **3.5. Eksekusi Full Experiment Pipeline (4.5 Jam Multi-Seed Training)**
Menjalankan seluruh siklus eksperimen komparatif (6 random seed, VRAM tracking, CUDA Event timing, Uji McNemar, Wilcoxon, Jackknife, Bootstrap CI, Cohen's d, Error Analysis, dan eksport `app.db`).

In [ ]:
# Menjalankan seluruh notebook eksperimen komparatif
%run Experiment_Notebook.ipynb


In [ ]:
# ==========================================
# 3. Konfigurasi Ngrok Static Tunnel (URL Permanen dengan Automatic Fallback)
# ==========================================
from pyngrok import ngrok
import os, sys

# 1. Masukkan NGROK AUTHTOKEN Anda dari https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""

# 2. NGROK STATIC DOMAIN permanen yang terhubung ke Frontend (config.js)
NGROK_STATIC_DOMAIN = "irritably-tipper-january.ngrok-free.dev"

if NGROK_AUTHTOKEN.strip():
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

# Bersihkan proses ngrok lama
os.system('pkill -9 ngrok > /dev/null 2>&1')
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass
ngrok.kill()

public_url = None
try:
    if NGROK_STATIC_DOMAIN.strip():
        print(f"Menghubungkan ke Static Domain: {NGROK_STATIC_DOMAIN}...")
        public_url = ngrok.connect(8000, domain=NGROK_STATIC_DOMAIN.strip())
    else:
        public_url = ngrok.connect(8000)
except Exception as e:
    err_msg = str(e)
    if 'ERR_NGROK_334' in err_msg or 'already online' in err_msg:
        print("\n⚠️ WARNING: Static domain sedang terikat pada sesi Ngrok lain.")
        print("💡 Solusi 1: Hentikan endpoint lama di https://dashboard.ngrok.com/endpoints")
        print("💡 Solusi 2: Menjalankan Fallback ke Random Dynamic URL Ngrok...\n")
        try:
            public_url = ngrok.connect(8000)
            print(f"✅ FALLBACK BERHASIL! URL Acak Sementara: {public_url}")
            print(f"   (Salin URL di atas jika ingin memasukkannya ke config.js)")
        except Exception as e2:
            print(f"❌ Kesalahan Ngrok Fallback: {e2}")
    else:
        print(f"❌ Kesalahan Ngrok: {e}")

if public_url:
    print("\n=======================================================")
    print("🎉 BACKEND GPU TESLA T4 BERHASIL LIVE ATAS NAMA:")
    print(f"👉 Base API URL       : {public_url}")
    print(f"👉 Prediction Endpoint : {public_url}/api/predict")
    print(f"👉 Health Check        : {public_url}/api/health")
    print("=======================================================\n")


In [ ]:
# ==========================================
# 4. Jalankan FastAPI Server pada GPU Tesla T4
# ==========================================
print("🚀 Memulai Server Uvicorn pada GPU Tesla T4...")
!python -m uvicorn backend.app.main:app --host 0.0.0.0 --port 8000